In [22]:
from pathlib import Path
import os
import sys
import glob

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("OlistAnalysis")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark

file_paths = glob.glob("../data/raw/*.csv")

dfs = {}
for path in file_paths:
    filename = os.path.splitext(os.path.basename(path))[0]
    clean_name = filename.replace("olist_", "").replace("_dataset", "")
    dfs[clean_name] = spark.read.csv(path, header=True, inferSchema=True, multiLine=True, escape='"')


customers = dfs["customers"]
geolocation = dfs["geolocation"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]
order_reviews = dfs["order_reviews"]
orders = dfs["orders"]
products = dfs["products"]
sellers = dfs["sellers"]
product_category_name_translation = dfs["product_category_name_translation"]

In [12]:
print(f"orders count: {orders.count()}")
orders.printSchema()
orders.show(18, truncate=False)

orders count: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+----------

STEP 1: Orders has 99,441 rows and 8 columns. All types are correct, so no casting is needed. One row appears to be one order (to verify later).

Note: Nulls are in order_delivered_carrier_date and order_delivered_customer_date. This makes sense for an invoice as it has not been delivered yet, but will need to verify.

In [21]:
orders.select(
    F.count(F.lit(1)).alias("total_rows"),
    F.count(F.col("customer_id")).alias("no_null_users"),
    F.countDistinct(F.col("customer_id")).alias("unique_users")
).show()

+----------+-------------+------------+
|total_rows|no_null_users|unique_users|
+----------+-------------+------------+
|     99441|        99441|       99441|
+----------+-------------+------------+

